# 🚀 TabDrift: Native Drifting Models for Tabular Data Generation (GPU Execution)

This notebook runs the complete end-to-end TabDrift pipeline on GPU (Kaggle).

- **Model**: Residual MLP TabDrift Generator (~10.6M parameters)
- **Sampling**: 1-Step ($K=1$) and 4-Step ($K=4$) Euler Drifting Sampler
- **Execution Time**: ~4 minutes for 500 epochs on Tesla T4 GPU!

In [ ]:
# 1. Clone Private GitHub Repository using Kaggle User Secrets
import os
from kaggle_secrets import UserSecretsClient

# Fetch secret token named GITHUB_TOKEN from Kaggle Add-ons -> Secrets
user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

# Set your GitHub username and repo name below
GITHUB_USER = "ahmed-fouad-lagha"
REPO_NAME = "tabsyn"

# Clone repository and change working directory
!git clone https://{github_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}

In [ ]:
# 2. Verify GPU Availability
!nvidia-smi

In [ ]:
# 3. Train TabDrift Generator for 500 Epochs on GPU (Takes ~4 minutes!)
!PYTHONPATH=. python tabsyn/drift_train.py \
    --dataname adult \
    --gpu 0 \
    --epochs 500 \
    --batch_size 4096 \
    --lr 1e-4 \
    --temperatures 0.1 0.5 1.0 2.0

In [ ]:
# 4. Generate 32,561 Synthetic Rows (1-NFE Direct Pass)
!PYTHONPATH=. python tabsyn/drift_sample.py \
    --dataname adult \
    --gpu 0 \
    --steps 1

In [ ]:
# 5. Evaluate 1-Step Machine Learning Efficacy (XGBoost ROC-AUC & F1)
!python eval/eval_mle.py --dataname adult --model tabdrift

In [ ]:
# 6. Generate 32,561 Synthetic Rows (4-Step Euler Drifting Refinement)
!PYTHONPATH=. python tabsyn/drift_sample.py \
    --dataname adult \
    --gpu 0 \
    --steps 4

In [ ]:
# 7. Evaluate 4-Step Machine Learning Efficacy (XGBoost ROC-AUC & F1)
!python eval/eval_mle.py --dataname adult --model tabdrift